# Notebook 1: Pre-Ingestion File Validation

**Purpose:** Validate JSON file before Auto Loader ingestion

**Checks:**
1. File exists and has valid size
2. File extension is .json
3. Content is actually JSON (not CSV/XML/text)
4. JSON can be parsed by Spark
5. Corrupt/malformed record detection
6. Schema structure
7. Array fields and explosion factors

**Output:** GO/CAUTION/STOP decision + Auto Loader config recommendations


## Configuration


In [ ]:
# ===== MODIFY THIS =====
FILE_PATH = "/Volumes/dev_automotive/landing/landing_raw/catalogs.json"

# Or use a widget (uncomment to use):
# dbutils.widgets.text("file_path", "/Volumes/dev_automotive/landing/landing_raw/catalogs.json", "JSON file path")
# FILE_PATH = dbutils.widgets.get("file_path").strip()


## Validation Start


In [ ]:
from pyspark.sql import functions as F

print("="*70)
print("FILE VALIDATION")
print("="*70)
print(f"File: {FILE_PATH}")
print()


## Step 1: File System Check


In [ ]:
print("1️⃣ Checking file...")

try:
    file_info = dbutils.fs.ls(FILE_PATH)[0]
    file_size_mb = round(file_info.size / (1024 * 1024), 2)
    file_name = file_info.name
    
    if not file_name.lower().endswith('.json'):
        print(f"   ⚠️  File extension is NOT .json: {file_name}")
        print(f"   → Will verify if content is actually JSON...")
        print()
    else:
        print(f"   ✅ File extension: .json")
    
    if file_size_mb == 0:
        print(f"   ❌ File is EMPTY (0 MB)")
        dbutils.notebook.exit("STOP: Empty file")
    else:
        print(f"   ✅ File size: {file_size_mb} MB")
        
except Exception as e:
    print(f"   ❌ File NOT found")
    print(f"   Error: {e}")
    dbutils.notebook.exit("STOP: File not found")

print()


## Step 2: Content Type Detection


In [ ]:
print("2️⃣ Checking file content type...")

try:
    raw_sample = spark.read.text(FILE_PATH).limit(5).collect()
    first_lines = [row.value for row in raw_sample if row.value.strip()]
    
    if not first_lines:
        print(f"   ❌ File appears empty or whitespace only")
        dbutils.notebook.exit("STOP: Empty content")
    
    first_line = first_lines[0].strip()
    file_type_detected = "UNKNOWN"
    
    if first_line.startswith('{') or first_line.startswith('['):
        file_type_detected = "JSON"
        print(f"   ✅ Content appears to be JSON")
    elif ',' in first_line and not first_line.startswith('{'):
        file_type_detected = "CSV"
        print(f"   ❌ Content appears to be CSV, NOT JSON")
        print(f"   First line: {first_line[:100]}...")
        dbutils.notebook.exit("STOP: Wrong format - CSV detected")
    elif first_line.startswith('<?xml') or first_line.startswith('<'):
        file_type_detected = "XML"
        print(f"   ❌ Content appears to be XML, NOT JSON")
        dbutils.notebook.exit("STOP: Wrong format - XML detected")
    elif 'PAR1' in first_line:
        file_type_detected = "PARQUET"
        print(f"   ❌ Content appears to be PARQUET, NOT JSON")
        dbutils.notebook.exit("STOP: Wrong format - Parquet detected")
    elif not any(char in first_line for char in ['{', '[', ',']):
        file_type_detected = "TEXT"
        print(f"   ❌ Content appears to be plain TEXT, NOT JSON")
        dbutils.notebook.exit("STOP: Wrong format - Plain text detected")
    else:
        print(f"   ⚠️  Cannot determine file type from content")
        print(f"   → Attempting to parse as JSON anyway...")
    
    print(f"   Detected format: {file_type_detected}")
except Exception as e:
    print(f"   ⚠️  Could not read raw content: {e}")
    print(f"   → Proceeding to JSON parse attempt...")

print()


## Step 3: JSON Parsing + Corrupt Record Detection


In [ ]:
print("3️⃣ Parsing as JSON...")

try:
    df = spark.read \
        .option("mode", "PERMISSIVE") \
        .option("columnNameOfCorruptRecord", "_corrupt_record") \
        .json(FILE_PATH)
    
    total_count = df.count()
    corrupt_count = 0
    if "_corrupt_record" in df.columns:
        corrupt_count = df.filter(F.col("_corrupt_record").isNotNull()).count()
    
    clean_count = total_count - corrupt_count
    
    print(f"   ✅ Successfully parsed as JSON")
    print(f"   ✅ Total records: {total_count:,}")
    print(f"   ✅ Clean records: {clean_count:,}")
    
    if corrupt_count > 0:
        corrupt_pct = round(corrupt_count/total_count*100, 1)
        print(f"   ⚠️  Corrupt records: {corrupt_count:,} ({corrupt_pct}%)")
        if corrupt_pct > 10:
            print(f"   🔴 HIGH corruption rate - investigate source")
        print(f"   → Enable badRecordsPath in Auto Loader")
        print()
        print(f"   Sample corrupt record:")
        corrupt_sample = df.filter(F.col("_corrupt_record").isNotNull()) \
            .select("_corrupt_record").limit(1).collect()
        if corrupt_sample:
            print(f"   {str(corrupt_sample[0][0])[:200]}...")
    else:
        print(f"   ✅ No corrupt records - all JSON is well-formed")
    
    if total_count == 0:
        print(f"   ❌ No records in file!")
        dbutils.notebook.exit("STOP: Empty JSON")
        
except Exception as e:
    print(f"   ❌ CANNOT parse as JSON")
    print(f"   Error: {str(e)[:200]}...")
    print(f"   → Verify file format at source")
    dbutils.notebook.exit("STOP: Cannot parse as JSON")

print()


## Step 4: Schema Analysis


In [ ]:
print("4️⃣ Schema:")
df.printSchema()
print()


## Step 5: Array Detection (Explosion Factor)


In [ ]:
print("5️⃣ Array check...")

array_fields = [f.name for f in df.schema.fields 
                if "array" in str(f.dataType).lower() 
                and f.name != "_corrupt_record"]

if array_fields:
    print(f"   ⚠️  Arrays found: {', '.join(array_fields)}")
    print()
    for arr_col in array_fields:
        try:
            avg_len = df.select(F.avg(F.size(arr_col))).first()[0]
            if avg_len:
                exploded_count = int(clean_count * avg_len)
                print(f"   Array: '{arr_col}'")
                print(f"   ├─ Avg length: {round(avg_len, 1)}")
                print(f"   ├─ Current rows: {clean_count:,}")
                print(f"   └─ After explosion: ~{exploded_count:,} rows ({round(avg_len, 1)}x)")
                print()
        except Exception as e:
            print(f"   ⚠️  Could not analyze array '{arr_col}': {e}")
    print(f"   💡 Recommendation: DON'T explode in Bronze")
    print(f"      → Keep nested structure intact")
    print(f"      → Do explosion in Silver layer with proper business logic")
else:
    print(f"   ✅ No arrays - flat structure")

print()


## Step 6: Sample Data Preview


In [ ]:
print("6️⃣ Sample data (first 5 rows):")
display(df.limit(5))


## Step 7: Final Decision


In [ ]:
print()
print("="*70)
print("VALIDATION DECISION")
print("="*70)

issues = []
warnings = []

if corrupt_count > 0:
    if corrupt_count / total_count > 0.1:
        issues.append(f"HIGH corrupt rate: {corrupt_count:,} records ({round(corrupt_count/total_count*100,1)}%)")
    else:
        warnings.append(f"Some corrupt records: {corrupt_count:,} ({round(corrupt_count/total_count*100,1)}%)")

if file_size_mb > 1000:
    warnings.append(f"Large file: {file_size_mb} MB")

if not file_name.lower().endswith('.json'):
    warnings.append(f"Non-standard extension: {file_name}")

if clean_count > 0:
    if len(issues) > 0:
        print("⚠️  GO WITH CAUTION - Critical issues found")
        print()
        for issue in issues:
            print(f"   🔴 {issue}")
        print()
    elif len(warnings) > 0:
        print("⚠️  GO - Minor warnings detected")
        print()
        for warning in warnings:
            print(f"   ⚠️  {warning}")
        print()
    else:
        print("✅ GO - File is healthy")
        print()
    
    print("Auto Loader Configuration Recommendations:")
    print("-" * 70)
    print("Use these settings in Notebook 2:")
    print()
    print("  spark.readStream \\")
    print("    .format('cloudFiles') \\")
    print("    .option('cloudFiles.format', 'json') \\")
    print("    .option('cloudFiles.schemaLocation', '<checkpoint_path>/schema') \\")
    print("    .option('cloudFiles.inferColumnTypes', 'true') \\")
    print("    .option('cloudFiles.schemaEvolutionMode', 'rescue') \\")
    print("    .option('rescuedDataColumn', '_rescued_data') \\")
    if corrupt_count > 0:
        print("    .option('badRecordsPath', '<checkpoint_path>/bad_records') \\")
    print("    .load(FILE_PATH) \\")
    print("    .withColumn('_ingestion_timestamp', F.current_timestamp()) \\")
    print("    .withColumn('_source_file', F.input_file_name())")
    print()
    if array_fields:
        print("Array Handling:")
        print(f"  • Keep arrays intact: {', '.join(array_fields)}")
        print("  • Handle explosion in Silver layer")
        print()
    print("✅ Ready to proceed to Notebook 2 (Auto Loader Ingestion)")
else:
    print("❌ STOP - No valid records found")
    print("  1. Verify file is correct format")
    print("  2. Check source system export process")
    print("  ❌ DO NOT proceed with ingestion")

print("="*70)


## Summary

**Next Steps:**
- If decision = **GO** → Proceed to Notebook 2 (Auto Loader)
- If decision = **CAUTION** → Review warnings, then proceed
- If decision = **STOP** → Fix issues before ingestion

